In [1]:
!pip install treys

In [2]:
import torch
import torch.optim as optim
import numpy as np
import os
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import deque
from treys import Card, Deck, Evaluator

In [3]:
class DQN(nn.Module):
    def __init__(self, state_dim=119, action_dim=3, hidden_dim=256):
        super().__init__()

        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)  # shape: [B, 3]


def select_action(model, state, valid_actions, epsilon, device):
    """
    Epsilon-greedy action selection with valid-action masking.
    """
    if random.random() < epsilon:
        return random.choice(valid_actions)

    state_t = torch.from_numpy(state).float().unsqueeze(0).to(device)

    with torch.no_grad():
        q_values = model(state_t)[0]  # [3]

        # Mask invalid actions
        mask = torch.full_like(q_values, -1e9)
        mask[valid_actions] = 0
        q_values = q_values + mask

        return int(torch.argmax(q_values).item())

In [4]:
class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done, next_valid_actions):
        self.buffer.append((state, action, reward, next_state, done, next_valid_actions))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)

        states, actions, rewards, next_states, dones, next_valid_actions = zip(*batch)

        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
            list(next_valid_actions),
        )

    def __len__(self):
        return len(self.buffer)

In [5]:
def train_dqn_step(
    q_net,
    target_net,
    replay_buffer,
    optimizer,
    batch_size,
    gamma,
    device
):
    if len(replay_buffer) < batch_size:
        return None

    states, actions, rewards, next_states, dones, next_valid_actions = replay_buffer.sample(batch_size)

    states = torch.tensor(states, device=device)
    actions = torch.tensor(actions, device=device).unsqueeze(1)
    rewards = torch.tensor(rewards, device=device)
    next_states = torch.tensor(next_states, device=device)
    dones = torch.tensor(dones, device=device)

    # Current Q(s, a)
    q_values = q_net(states)
    chosen_q = q_values.gather(1, actions).squeeze(1)

    with torch.no_grad():
        next_q_values = target_net(next_states)

        # Mask invalid next actions
        for i, valid in enumerate(next_valid_actions):
            invalid = [a for a in range(3) if a not in valid]
            next_q_values[i, invalid] = -1e9

        max_next_q = next_q_values.max(dim=1).values

        target = rewards + gamma * (1.0 - dones) * max_next_q

    loss = F.smooth_l1_loss(chosen_q, target)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
    optimizer.step()

    return loss.item()

In [16]:
class PokerEnv:
    def __init__(self, num_players=6, starting_stack=1000, small_blind=10, big_blind=20):
        self.num_players = num_players
        self.starting_stack = starting_stack
        self.small_blind = small_blind
        self.big_blind = big_blind
        self.evaluator = Evaluator()
        self.reset()

    def reset(self, custom_stacks=None): # Added custom_stacks parameter
        self.deck = Deck()
        self.board = []
        self.hands = [self.deck.draw(2) for _ in range(self.num_players)]
        if custom_stacks:
            self.stacks = custom_stacks[:]
        else:
            self.stacks = [self.starting_stack] * self.num_players
        self.active = [True] * self.num_players
        self.pot = 0
        self.button = 0 # Player 0 is dealer

        self.bets = [0] * self.num_players
        self.total_invested = [0] * self.num_players
        self.acted = [False] * self.num_players

        self._post_blinds()

        # SB is button + 1, BB is button + 2. Under the Gun is button + 3
        self.current_player = (self.button + 3) % self.num_players

        self.stage = 0 # 0: Preflop, 1: Flop, 2: Turn, 3: River
        self.street_finished = False
        self.hand_over = False
        self.winners = []
        self.reward = [0.0] * self.num_players

        return self.current_player, self._get_state(self.current_player)

    def _post_blinds(self):
        sb_idx = (self.button + 1) % self.num_players
        bb_idx = (self.button + 2) % self.num_players

        sb = min(self.small_blind, self.stacks[sb_idx])
        self.stacks[sb_idx] -= sb
        self.bets[sb_idx] += sb
        self.total_invested[sb_idx] += sb
        self.pot += sb

        bb = min(self.big_blind, self.stacks[bb_idx])
        self.stacks[bb_idx] -= bb
        self.bets[bb_idx] += bb
        self.total_invested[bb_idx] += bb
        self.pot += bb

    def _get_vector(self, cards):
        vec = np.zeros(52, dtype=np.float32)
        if type(cards) is int:
            cards = [cards]

        for c in cards:
            rank = Card.get_rank_int(c)
            suit = Card.get_suit_int(c)
            # suit is 1, 2, 4, 8 -> map to 0, 1, 2, 3
            suit_map = {1:0, 2:1, 4:2, 8:3}
            idx = rank * 4 + suit_map.get(suit, 0)
            vec[idx] = 1.0
        return vec

    def _get_state(self, player_idx):
        hand_vec = self._get_vector(self.hands[player_idx])
        board_vec = self._get_vector(self.board)

        # 1. Existing Chip / Player Stats
        normalization_factor = self.starting_stack * self.num_players
        stack = self.stacks[player_idx] / normalization_factor
        highest_bet = max(self.bets)
        to_call_raw = max(0, highest_bet - self.bets[player_idx])
        to_call = to_call_raw / normalization_factor
        pot_size = self.pot / normalization_factor
        active_opponents = (sum(self.active) - 1) / max(1, self.num_players - 1)

        # 2. NEW: Game Stage (One-hot encoded)
        stage_vec = np.zeros(4, dtype=np.float32)
        stage_vec[min(self.stage, 3)] = 1.0 # 0: Preflop, 1: Flop, 2: Turn, 3: River

        # 3. NEW: Position Relative to Button (One-hot encoded)
        pos_vec = np.zeros(self.num_players, dtype=np.float32)
        dist_to_button = (player_idx - self.button) % self.num_players
        pos_vec[dist_to_button] = 1.0

        # 4. NEW: Pot Odds
        # to_call / (pot + to_call). Added 1e-9 to prevent division by zero.
        pot_odds = to_call_raw / (self.pot + to_call_raw + 1e-9)

        # Combine everything into a single 118-dimension array
        state = np.concatenate([
            hand_vec,           # 52 dims
            board_vec,          # 52 dims
            stage_vec,          # 4 dims
            pos_vec,            # 5 dims
            [stack, to_call, pot_size, active_opponents, pot_odds] # 5 dims
        ])

        return state

    def get_valid_actions(self, player_idx):
        highest_bet = max(self.bets)
        to_call = max(0, highest_bet - self.bets[player_idx])

        valid = [0, 1] # Fold, Call/Check

        if self.stacks[player_idx] > to_call:
            valid.append(2) # Raise

        return valid

    def _next_active_player(self, current):
        nxt = (current + 1) % self.num_players
        while not self.active[nxt] or self.stacks[nxt] == 0:
            nxt = (nxt + 1) % self.num_players
            if nxt == current:
                return -1 # No other active players with non-zero stack
        return nxt

    def _is_street_finished(self):
        highest_bet = max(self.bets)
        active_players_with_chips = 0
        for i in range(self.num_players):
            if self.active[i]:
                if self.stacks[i] > 0:
                    active_players_with_chips += 1
                if self.stacks[i] > 0 and (not self.acted[i] or self.bets[i] < highest_bet):
                    return False
        return True

    def step(self, action):
        if self.hand_over:
            raise ValueError("Hand is already over")

        p = self.current_player
        self.acted[p] = True

        highest_bet = max(self.bets)
        to_call = max(0, highest_bet - self.bets[p])

        if action == 0: # Fold
            self.active[p] = False

        elif action == 1: # Call / Check
            call_amount = min(to_call, self.stacks[p])
            self.stacks[p] -= call_amount
            self.bets[p] += call_amount
            self.total_invested[p] += call_amount
            self.pot += call_amount

        elif action == 2: # Raise
            call_amount = min(to_call, self.stacks[p])
            pot_after_call = self.pot + call_amount
            raise_amount = min(pot_after_call // 2, self.stacks[p] - call_amount)
            if raise_amount <= 0:
                raise_amount = min(self.big_blind, self.stacks[p] - call_amount)

            total_bet = call_amount + raise_amount
            self.stacks[p] -= total_bet
            self.bets[p] += total_bet
            self.total_invested[p] += total_bet
            self.pot += total_bet

            # Reset acted flags for others because of the raise
            for i in range(self.num_players):
                if i != p and self.active[i] and self.stacks[i] > 0:
                    self.acted[i] = False

        # Check if hand ended by everyone folding except 1
        active_count = sum(self.active)
        if active_count == 1:
            self.hand_over = True
            for i in range(self.num_players):
                if self.active[i]:
                    self.winners = [i]
            self._compute_rewards()
            done = True
            return self.current_player, self._get_state(self.current_player), self.reward.copy(), done

        self.street_finished = self._is_street_finished()

        if self.street_finished:
            active_with_chips = sum(1 for i in range(self.num_players) if self.active[i] and self.stacks[i] > 0)

            if active_with_chips <= 1 and active_count > 1: # All in situation, not everyone folded
                while self.stage < 3:
                    self._next_stage()
                self.hand_over = True
                self._evaluate_showdown()
            else:
                self._next_stage()
                if self.stage > 3: # Showdown
                    self.hand_over = True
                    self._evaluate_showdown()
                else: # Next street setup
                    self.acted = [False] * self.num_players
                    self.bets = [0] * self.num_players
                    # First active player after button
                    nxt = self._next_active_player(self.button)
                    self.current_player = nxt if nxt != -1 else self.current_player
        else:
            nxt = self._next_active_player(self.current_player)
            if nxt != -1:
                self.current_player = nxt

        done = self.hand_over
        if done and len(self.winners) == 0:
            # edge case fallback
            self._compute_rewards()

        state = self._get_state(self.current_player) if not done else None

        return self.current_player, state, self.reward.copy(), done

    def _next_stage(self):
        self.stage += 1
        num_draw = 3 if self.stage == 1 else 1 if self.stage <= 3 else 0
        if num_draw > 0:
            cards = self.deck.draw(num_draw)
            if type(cards) is int:
                 self.board.append(cards)
            else:
                 self.board.extend(cards)

    def _evaluate_showdown(self):
        # We ensure river is dealt before calling this
        if len(self.board) < 5:
            pass

        best_score = float('inf')
        best_players = []

        for i in range(self.num_players):
            if self.active[i]:
                try:
                    score = self.evaluator.evaluate(self.board, self.hands[i])
                    if score < best_score:
                        best_score = score
                        best_players = [i]
                    elif score == best_score:
                        best_players.append(i)
                except Exception as e:
                    # In case of treys failure, just skip
                    pass

        self.winners = best_players
        self._compute_rewards()

    def _compute_rewards(self):
        # Simple pot distribution
        num_winners = len(self.winners)
        if num_winners == 0:
            return

        win_amount = self.pot / num_winners

        for i in range(self.num_players):
            if i in self.winners:
                self.reward[i] = (win_amount - self.total_invested[i]) / self.starting_stack
            else:
                self.reward[i] = -self.total_invested[i] / self.starting_stack


In [7]:
def train_dqn(
    episodes=50000,
    lr=1e-4,
    gamma=0.99,
    batch_size=128,
    buffer_size=100000,
    target_update=1000,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=30000,
    save_path="dqn_hero.pth"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = PokerEnv(num_players=6)

    q_net = DQN(state_dim=119, action_dim=3, hidden_dim=256).to(device)
    target_net = DQN(state_dim=119, action_dim=3, hidden_dim=256).to(device)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(q_net.parameters(), lr=lr)
    replay_buffer = ReplayBuffer(capacity=buffer_size)

    global_step = 0
    rewards_history = []

    for ep in range(episodes):
        current_player, state = env.reset()
        done = False

        episode_rewards = [0.0 for _ in range(env.num_players)]

        while not done:
            valid_actions = env.get_valid_actions(current_player)

            epsilon = epsilon_end + (epsilon_start - epsilon_end) * np.exp(
                -global_step / epsilon_decay
            )

            action = select_action(
                q_net,
                state,
                valid_actions,
                epsilon,
                device
            )

            next_player, next_state, rewards, done = env.step(action)

            hero_reward = rewards[0]

            if done:
                replay_buffer.push(
                    state,
                    action,
                    hero_reward,
                    np.zeros_like(state),
                    done,
                    []
                )
            else:
                next_valid_actions = env.get_valid_actions(next_player)

                replay_buffer.push(
                    state,
                    action,
                    hero_reward,
                    next_state,
                    done,
                    next_valid_actions
                )

            loss = train_dqn_step(
                q_net=q_net,
                target_net=target_net,
                replay_buffer=replay_buffer,
                optimizer=optimizer,
                batch_size=batch_size,
                gamma=gamma,
                device=device
            )

            if global_step % target_update == 0:
                target_net.load_state_dict(q_net.state_dict())

            state = next_state
            current_player = next_player
            global_step += 1

        rewards_history.append(rewards[0])

        if ep % 100 == 0:
            avg_reward = np.mean(rewards_history[-100:]) if rewards_history else 0
            print(
                f"Episode {ep} | "
                f"epsilon={epsilon:.3f} | "
                f"avg hero reward={avg_reward:.4f}"
            )

        if ep % 5000 == 0 and ep > 0:
            torch.save(q_net.state_dict(), save_path)

    torch.save(q_net.state_dict(), save_path)
    print(f"Saved DQN model to {save_path}")

In [8]:
train_dqn()

Episode 0 | epsilon=1.000 | avg hero reward=2.7550
Episode 100 | epsilon=0.962 | avg hero reward=-0.0432
Episode 200 | epsilon=0.924 | avg hero reward=-0.0884
Episode 300 | epsilon=0.889 | avg hero reward=-0.0051
Episode 400 | epsilon=0.854 | avg hero reward=0.1458
Episode 500 | epsilon=0.821 | avg hero reward=0.1062
Episode 600 | epsilon=0.790 | avg hero reward=0.0980
Episode 700 | epsilon=0.758 | avg hero reward=-0.0446
Episode 800 | epsilon=0.727 | avg hero reward=0.0739
Episode 900 | epsilon=0.697 | avg hero reward=-0.0562
Episode 1000 | epsilon=0.668 | avg hero reward=-0.0123
Episode 1100 | epsilon=0.638 | avg hero reward=0.0173
Episode 1200 | epsilon=0.611 | avg hero reward=0.0553
Episode 1300 | epsilon=0.584 | avg hero reward=0.1531
Episode 1400 | epsilon=0.558 | avg hero reward=-0.0976
Episode 1500 | epsilon=0.534 | avg hero reward=-0.0348
Episode 1600 | epsilon=0.511 | avg hero reward=0.1549
Episode 1700 | epsilon=0.488 | avg hero reward=0.1879
Episode 1800 | epsilon=0.467 | a

In [9]:

from torch.distributions import Categorical

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim=108, action_dim=3, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)

    def forward(self, state, valid_actions=None):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)

        if valid_actions is not None:
            # Create a mask of negative infinity
            mask = torch.full(logits.shape, -float('inf')).to(logits.device)
            # Set valid action indices to 0 so they don't change the logit value
            mask[0, valid_actions] = 0
            logits = logits + mask

        return F.softmax(logits, dim=-1)

    def select_action(self, state, valid_actions):
        device = next(self.parameters()).device
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        # Pass valid_actions to forward pass
        probs = self.forward(state, valid_actions)

        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action), m.entropy()

class ActorCritic(nn.Module):
    def __init__(self, state_dim=119, action_dim=3, hidden_dim=256):
        super(ActorCritic, self).__init__()
        # Shared features
        self.fc1 = nn.Linear(state_dim, hidden_dim)

        # Actor head
        self.actor_fc = nn.Linear(hidden_dim, hidden_dim)
        self.actor_out = nn.Linear(hidden_dim, action_dim)

        # Critic head
        self.critic_fc = nn.Linear(hidden_dim, hidden_dim)
        self.critic_out = nn.Linear(hidden_dim, 1)

    def forward(self, state, valid_actions=None):
        x = F.relu(self.fc1(state))

        # Actor
        actor_x = F.relu(self.actor_fc(x))
        logits = self.actor_out(actor_x)

        if valid_actions is not None:
            mask = torch.full(logits.shape, -float('inf')).to(logits.device)
            mask[0, valid_actions] = 0
            logits = logits + mask

        probs = F.softmax(logits, dim=-1)

        # Critic
        critic_x = F.relu(self.critic_fc(x))
        value = self.critic_out(critic_x)

        return probs, value

    def select_action(self, state, valid_actions):
        device = next(self.parameters()).device
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs, value = self.forward(state, valid_actions)

        m = Categorical(probs)
        action = m.sample()

        return action.item(), m.log_prob(action), m.entropy(), value

In [14]:

hero_model = "dqn_hero.pth"
villain_model = "hero_agent_2.pth"

def test_average(num_games=1000):
    env = PokerEnv(num_players=6) # Changed num_players to 6 to match model state_dim

    hero = DQN(state_dim=119, hidden_dim=256)
    # Correctly initialize villains to match the number of players in the environment
    villains = [ActorCritic(state_dim=119, hidden_dim=256) for _ in range(env.num_players - 1)]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Define device once

    if os.path.exists(hero_model):
        weights = torch.load(hero_model, weights_only=True)
        # The padding logic for 118-dim models is not strictly needed if the model is already 119-dim
        # However, keeping it as it was in case of other intended usages.
        if 'fc1.weight' in weights and weights['fc1.weight'].shape[1] == 118:
            w = weights['fc1.weight']
            padded_w = torch.cat([w, torch.zeros(w.shape[0], 1).to(w.device)], dim=1)
            weights['fc1.weight'] = padded_w
        hero.load_state_dict(weights)
        print(f"Loaded saved weights for Hero ({hero_model}).")
    else:
        print(f"Hero model {hero_model} not found.")

    hero.to(device) # Move hero model to the selected device

    if os.path.exists(villain_model):
        print(f"Loading pre-trained {villain_model} into Villains...")
        try:
            weights = torch.load(villain_model, weights_only=True)
            # The padding logic for 118-dim models is not strictly needed if the model is already 119-dim
            # However, keeping it as it was in case of other intended usages.
            if 'fc1.weight' in weights and weights['fc1.weight'].shape[1] == 118:
                w = weights['fc1.weight']
                padded_w = torch.cat([w, torch.zeros(w.shape[0], 1).to(w.device)], dim=1)
                weights['fc1.weight'] = padded_w
            for v in villains:
                v.load_state_dict(weights)
                v.to(device) # Move each villain model to the selected device
        except Exception as e:
            print(f"Failed to load {villain_model}:", e)
    else:
        print(f"Villain model {villain_model} not found.")

    hero.eval()
    for v in villains:
        v.eval()

    agents = [hero] + villains

    print(f"=========================================")
    print(f"Starting {num_games} Sample 6-Player Poker Games")
    print(f"=========================================")

    total_hero_reward = 0.0
    total_villain_reward = 0.0

    hero_wins = 0
    villain_wins = 0
    ties = 0

    for game in range(1, num_games + 1):
        try:
            current_player, state = env.reset()
        except Exception as e:
            print("Failed to initialize game:", e)
            continue

        done = False

        while not done:
            valid_actions = env.get_valid_actions(current_player)
            # device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Moved device definition outside the loop

            with torch.no_grad():
                agent = agents[current_player]
                if isinstance(agent, DQN):
                    # For DQN (hero), call the global select_action function
                    action = select_action(
                        agent, # This is the DQN model
                        state,
                        valid_actions,
                        epsilon=0.0, # Use greedy action selection for evaluation
                        device=device
                    )
                elif isinstance(agent, ActorCritic):
                    # For ActorCritic (villains), call its select_action method
                    action, _, _, _ = agent.select_action(state, valid_actions)
                else:
                    raise TypeError(f"Unknown agent type: {type(agent)}")

            try:
                current_player, next_state, rewards, done = env.step(action);
            except Exception as e:
                done = True
                rewards = [0] * env.num_players # Adjust rewards list size based on actual num_players

            state = next_state

        # Add to total rewards
        for i in range(env.num_players): # Iterate over actual num_players
            absolute_reward = rewards[i] * env.starting_stack * env.num_players
            if i == 0:
                total_hero_reward += absolute_reward
            else:
                total_villain_reward += absolute_reward

        if len(env.winners) == 1:
            if env.winners[0] == 0:
                hero_wins += 1
            else:
                villain_wins += 1
        elif len(env.winners) > 1:
            if 0 in env.winners:
                hero_wins += 1 / len(env.winners)
                villain_wins += (len(env.winners) - 1) / len(env.winners)
            else:
                villain_wins += 1
        else:
            ties += 1

        if game % 100 == 0:
            print(f"Completed {game}/{num_games} games...")

    avg_hero_reward = total_hero_reward / num_games
    # Adjust villain reward calculation for 5 villains, not 1, when num_players=6 (1 hero + 5 villains)
    avg_villain_reward = total_villain_reward / (num_games * (env.num_players - 1))
    sum_villains_avg_per_game = total_villain_reward / num_games

    print("\n=========================================")
    print(f"Results after {num_games} games:")
    print("=========================================")
    print(f"Hero Wins: {hero_wins:.2f}")
    print(f"Villain Wins (total for all {env.num_players - 1}): {villain_wins:.2f}") # Adjusted for dynamic num_players
    print(f"Ties: {ties}")
    print(f"-----------------------------------------")
    print(f"Total Hero Reward: ${total_hero_reward:.2f}")
    print(f"Total Villains Reward: ${total_villain_reward:.2f}")
    print(f"-----------------------------------------")
    print(f"Average Hero Reward per game: ${avg_hero_reward:.2f}")
    print(f"Average total Villains Reward per game: ${sum_villains_avg_per_game:.2f}")
    print(f"Average Reward per Villain per game: ${avg_villain_reward:.2f}")

if __name__ == "__main__":
    test_average()


Loaded saved weights for Hero (dqn_hero.pth).
Loading pre-trained hero_agent_2.pth into Villains...
Starting 1000 Sample 6-Player Poker Games
Completed 100/1000 games...
Completed 200/1000 games...
Completed 300/1000 games...
Completed 400/1000 games...
Completed 500/1000 games...
Completed 600/1000 games...
Completed 700/1000 games...
Completed 800/1000 games...
Completed 900/1000 games...
Completed 1000/1000 games...

Results after 1000 games:
Hero Wins: 154.32
Villain Wins (total for all 5): 845.68
Ties: 0
-----------------------------------------
Total Hero Reward: $-10350.30
Total Villains Reward: $10350.30
-----------------------------------------
Average Hero Reward per game: $-10.35
Average total Villains Reward per game: $10.35
Average Reward per Villain per game: $2.07


In [23]:
def print_cards(cards, prefix=""): # Added print_cards function
    card_strings = [Card.int_to_str(c) for c in cards]
    print(f"{prefix} {' '.join(card_strings)}")

def play_vs_bot():
    env = PokerEnv(num_players=6)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load villain (DQN) models - using dqn_hero.pth for villains now
    villains = [DQN(state_dim=119, hidden_dim=256).to(device) for _ in range(env.num_players - 1)]

    if os.path.exists(hero_model): # Loading dqn_hero.pth for villains
        print(f"Loading hero model {hero_model} into Villains...")
        weights = torch.load(hero_model, weights_only=True)
        if 'fc1.weight' in weights and weights['fc1.weight'].shape[1] == 118:
            w = weights['fc1.weight']
            padded_w = torch.cat([w, torch.zeros(w.shape[0], 1).to(w.device)], dim=1)
            weights['fc1.weight'] = padded_w
        for v in villains:
            v.load_state_dict(weights)
    else:
        print(f"DQN model {hero_model} not found. Villains will use untrained models.")

    for v in villains:
        v.eval()

    # Hero (Player 0) is human input, others are DQN villains
    agents = [None] + villains

    print("=========================================")
    print("🎮 You are the Hero (Player 0)")
    print("Actions: 0 = Fold, 1 = Call/Check, 2 = Raise")
    print("=========================================")

    current_player, state = env.reset(custom_stacks=[1000, 300, 300, 300, 300, 300])

    print("\n--- Pre-flop ---")
    print_cards(env.hands[0], "Your Hand:") # Show human hero's hand

    stage_names = ["Pre-flop", "Flop", "Turn", "River"]
    current_stage = 0

    done = False

    while not done:

        # Stage update
        if env.stage > current_stage:
            current_stage = env.stage

            print(f"\n--- {stage_names[current_stage]} ---")
            print_cards(env.board, "Board:")
            print(f"Pot: ${env.pot}")

            print("\nHands:")
            for i in range(6):
                name = "You" if i == 0 else f"Villain {i}"
                print_cards(env.hands[i], f"{name}:")

            print("")  # spacing

        valid_actions = env.get_valid_actions(current_player)
        action_names = {0: "Fold", 1: "Call/Check", 2: "Raise"}

        if current_player == 0:
            # HUMAN INPUT
            print("\nYour turn")
            print(f"Valid actions: {[action_names[a] for a in valid_actions]}")
            print(f"To call: {max(env.bets) - env.bets[0]}")
            print(f"Your stack: {env.stacks[0]}")

            while True:
                try:
                    action = int(input("Enter action (0/1/2): "))
                    if action in valid_actions:
                        break
                    else:
                        print("Invalid action.")
                except ValueError:
                    print("Enter a number (0,1,2)")
        else:
            # DQN BOT ACTION
            with torch.no_grad():
                agent = agents[current_player]
                # For DQN (villain), call the global select_action function
                action = select_action(
                    agent, # This is the DQN model
                    state,
                    valid_actions,
                    epsilon=0.0, # Greedy action for evaluation
                    device=device
                )
            print(f"Villain {current_player} chooses: {action_names[action]}")

        current_player, state, rewards, done = env.step(action)

    # --- Game Over ---
    print("\n=========================================")
    print("Game Over")

    if len(env.winners) > 0:
        winner_names = ["You" if w == 0 else f"Villain {w}" for w in env.winners]
        print("Winners:", ", ".join(winner_names))

    print("\nFinal Rewards:")
    for i in range(6):
        name = "You" if i == 0 else f"Villain {i}"
        val = rewards[i] * env.starting_stack * env.num_players
        print(f"{name}: {val:.2f}")

    print("\nFinal Board:")
    print_cards(env.board)

    print("\nHands:")
    for i in range(6):
        name = "You" if i == 0 else f"Villain {i}"
        print_cards(env.hands[i], f"{name}:")

play_vs_bot()

Loading hero model dqn_hero.pth into Villains...
🎮 You are the Hero (Player 0)
Actions: 0 = Fold, 1 = Call/Check, 2 = Raise

--- Pre-flop ---
Your Hand: Jc 8h
Villain 3 chooses: Call/Check
Villain 4 chooses: Call/Check
Villain 5 chooses: Call/Check

Your turn
Valid actions: ['Fold', 'Call/Check', 'Raise']
To call: 20
Your stack: 1000
Enter action (0/1/2): 0
Villain 1 chooses: Raise
Villain 2 chooses: Raise
Villain 3 chooses: Raise
Villain 4 chooses: Call/Check
Villain 5 chooses: Call/Check
Villain 1 chooses: Call/Check
Villain 2 chooses: Call/Check

Game Over
Winners: Villain 1

Final Rewards:
You: 0.00
Villain 1: 7200.00
Villain 2: -1800.00
Villain 3: -1800.00
Villain 4: -1800.00
Villain 5: -1800.00

Final Board:
 Th 4s 9s 4c Ks

Hands:
You: Jc 8h
Villain 1: Ts Qs
Villain 2: 7s 3s
Villain 3: 6h 8s
Villain 4: 7h Ah
Villain 5: 5d 5h
